# Step 22.4 — MHA → MQA / GQA：减少 KV 头的数量

## 先建立独立工作副本（以下只是命令参考，本次未执行）

在仓库根目录先同步母版：

```bash
git pull
```

再创建不覆盖已有文件的时间戳副本（仓库应已有 notebooks/work 目录）：

```bash
cp -n notebooks/100_step22_4_mqa_gqa_attention.ipynb notebooks/work/100_step22_4_mqa_gqa_attention_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

只在工作副本中填写练习或未来运行。`cp -n` 在同名目标存在时不覆盖；同一秒再次操作也不会覆盖已有内容。2026-11-17 教学修订：**22.4 累计 1/2＝历史修正 0/2＋本次教学修订 1 次**。本次只补连续分组、shape、causal 行和两条计算路径的解读；原函数、观察源码与练习保留，未运行任何课程代码。

**执行边界：**所有代码单元 execution_count 为 null、outputs 为空。本次不执行 notebook、MLX 实验、测试、训练、验证、采样、语料读取/下载或 checkpoint 操作。下面的短实验是将来可观察的安排，不是已观察结果。MLX API 基线固定 v0.32.2。

## 1. 为什么需要这一节

Multi-head attention（MHA）为每个 query head 配一个独立 K/V head。Multi-query attention（MQA）让所有 query heads 共享一组 K/V；grouped-query attention（GQA）在中间取折衷：多组 query heads 各共享一个 K/V head。

动机尤其来自 autoregressive decoding 的 KV cache。每步只生成一个 token，却需要读取历史 K/V；减少 KV heads 能减少逻辑缓存容量和相关带宽需求。它改变的是架构与参数，不是纯粹换一个执行算法。

## 2. 先备数学与 Python / MLX 语义

令 Hq 为 query heads、Hkv 为 KV heads，要求 Hq % Hkv=0。每组 g=Hq/Hkv 个 query heads 使用同一个 KV head。MHA: Hkv=Hq；MQA: Hkv=1；GQA: 1<Hkv<Hq。

每层 K/V cache 的元素数为 2BTHkv d，N 层、每元素 s bytes 时逻辑容量为 2NBTHkv d s。这个公式没有包含模型权重、临时 attention scores、allocator 保留、碎片或系统 RAM。GQA 减少 KV 数量不等于消除 query heads 的打分计算。

## 3. 纸笔走一遍（讲解例子，不是练习答案）

B=1，N=2，T=64，d=16，Hq=4，float16 每元素 2 bytes。MHA Hkv=4：逻辑 cache 为 32768 bytes；GQA Hkv=2：16384 bytes；MQA Hkv=1：8192 bytes。只是在给定假设下的手算，不是硬件测量。

GQA Hq=4,Hkv=2 的映射为 query heads [0,1]→KV head 0，[2,3]→KV head 1。不能交错成 [0,2] 与 [1,3] 又声称是同一 head 排列。

### 3.1 把连续分组放到 repeat 的 head 轴

沿用 Hq=4、Hkv=2，g=Hq//Hkv=2。这里每一个 K₀ 或 K₁ 都代表该 KV head 的**整段 T×d 矩阵**，不是某一个时间位置：

| query head | 使用的原 K/V head | repeat 后同编号位置放什么 |
|---:|---:|---|
| 0 | 0 | K₀、V₀ |
| 1 | 0 | K₀、V₀ |
| 2 | 1 | K₁、V₁ |
| 3 | 1 | K₁、V₁ |

原代码的 `mx.repeat(k,2,axis=1)` 沿 head 轴把**每个 head 连续重复两次**：[K₀,K₁] → [K₀,K₀,K₁,K₁]。它不是把整份列表拼接两遍成为 [K₀,K₁,K₀,K₁]，也不沿 time 轴复制 token。V 使用同样的映射，不能只重复 K 而让 V 的分组错位。

query heads 0、1 虽共享 K₀/V₀，Q₀、Q₁ 仍是各自的向量，算出的分数和输出可以不同。共享 KV 不等于共享 query，也不等于把两个 query 输出复制成相同结果。

## 4. 按现有代码逐步跟踪 shape

一般 Q 为 (B,Hq,Tq,d)，K/V 为 (B,Hkv,Tk,d)，但本节 `gqa_reference` **只接受 Tq=Tk=T 的 causal self-attention**。下面使用原观察单元的 B=1、Hq=4、Hkv=2、T=3、d=2：

| 源码中的对象／操作 | Shape | 哪条轴在做什么 |
|---|---|---|
| 原 q | (1,4,3,2) | 4 个 query heads，各有 3 个位置、2 个特征 |
| 原 k、v | (1,2,3,2) | 每组共享的 2 个 KV heads |
| repeat(k/v,2,axis=1) | (1,4,3,2) | 只扩展 head 轴，保持每段 time/feature 顺序 |
| repeated_k.transpose(0,1,3,2) | (1,4,2,3) | 交换最后两轴，为每个 head 的矩阵乘法准备 (d,T) |
| q @ repeated_kᵀ，再乘 d⁻¹ᐟ² | (1,4,3,3) | 每个 query 位置对所有 key 位置打分 |
| allowed 与 mx.where 后的 scores | (3,3) mask 广播到 (1,4,3,3) | 每个 batch/head 共用同一时间可见性规则 |
| softmax(scores,axis=-1) | (1,4,3,3) | 每个 query 行沿 key 轴归一化 |
| 权重 @ repeated_v | (1,4,3,2) | 每个 query/head 得到一个 d=2 的加权内容向量 |

前导的 B/head 轴让这些小矩阵运算彼此独立，不是在不同 head 之间做点积。输出还没有合并 heads 或经过输出投影；helper 只返回 head-major attention 结果。

### 4.1 等长 causal mask 怎样广播？

原代码先构造两个索引数组：

```text
allowed = mx.arange(T)[:, None] >= mx.arange(T)[None, :]
scores = mx.where(allowed, scores, float("-inf"))
```

左侧 shape (3,1) 是 query 行索引 i，右侧 (1,3) 是 key 列索引 j；广播比较 i≥j，得到 (3,3)：

| query i ＼ key j | 0 | 1 | 2 |
|---:|---|---|---|
| 0 | True | False | False |
| 1 | True | True | False |
| 2 | True | True | True |

把 (3,3) 与 scores (1,4,3,3) 对齐时，相当于 mask 带有前导 (1,1) 轴，同一张表应用到全部 query heads。True 保留分数，False 改成 −Inf；softmax 的 exp(−Inf)=0 才把未来位置排除。用 0 替换分数并不是屏蔽。

这一行先处理可见性，再沿最后的 key 轴 softmax；不是沿 head 轴归一化。这里各行至少允许自身位置，所以不会产生全被屏蔽的行。Tq≠Tk 的 decode 对齐不由这段方阵规则自动解决。

### 4.2 只手算一行：query head 2、时间位置 1

直接从下方原观察单元的 arange/reshape 定义读数，不调用函数。q 按 head/time/feature 顺序排布，因此 `q[0,2,1]` 是 [14,15]/24。head 2 属于第二组，使用原 KV head **1**，不是 KV head 0。

| key 位置 j | 原 k[0,1,j] | 点积除以 √2 后的 score | i=1 是否允许 |
|---:|---|---|---|
| 0 | [6,7]/12 | (14×6+15×7)/(24×12×√2)=189/(288√2) | 是 |
| 1 | [8,9]/12 | (14×8+15×9)/(24×12×√2)=247/(288√2) | 是 |
| 2 | [10,11]/12 | (14×10+15×11)/(24×12×√2)=305/(288√2) | 否 |

记前两项为 a、b，mask 后为 [a,b,−Inf]。这行权重为 [α,β,0]，其中：

$$
\alpha=\frac{e^a}{e^a+e^b},\qquad
\beta=\frac{e^b}{e^a+e^b},\qquad \alpha+\beta=1.
$$

尽管 j=2 的原分数更高，它仍不可见，不能进入归一化分母。最后只取同组 V 的允许位置：

$$
\mathrm{out}[0,2,1]
=\alpha\,[6,7]/6+\beta\,[8,9]/6,
$$

输出是长度 2 的向量，而不是某一个 token ID。这里的分数与分式是按给定构造做的理想实数纸算；实际 float32 计算会有舍入。本次没有生成 reference/fast 的数值，也没有预填差值或运行结果。

### 4.3 减少的容量，不等于全部工作量按相同比例减少

第 3 节的 32768/16384/8192 bytes 例子是**假设已存紧凑 KV cache、dtype=float16**时的逻辑容量。本页后面的 float32 等长参考计算没有实现 cache，不能拿这个公式当作它的实际分配或测量。

Hq 保持 4 时，reference 的 scores 仍为 (1,4,3,3)，四个 query heads 都要打分与加权。一般打分/加权的主要乘加规模仍随 B·Hq·T²·d 增长，不会仅因 Hkv 从 4 降为 2 就全部减半。KV 投影、紧凑缓存和相关读取需求则随 KV heads 数改变。

显式 repeat 扩大了本参考中 K/V 的逻辑数组形状，scores 也要纳入考虑；不能把它包装成保留全部紧凑缓存优势的生产实现。真实临时分配、峰值内存、带宽与速度取决于执行路径，本次没有测量。

### 4.4 保留原 reference：用于对照上述定义

下面的源码足够短，保持原样。它核对 K/V 同 shape、B/T/d 与 Q 匹配，以及 Hq 能被正的 Hkv 整除；随后按连续 head 组 repeat、打分、mask、归一化、加权。它不创建投影参数，不维护 cache，也不是覆盖所有非法 dtype/非有限输入的通用验证器。

原 [MiniGPT 的 MHA](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/479bd59af3956c74cbad21787c7fdf8ff17eeda3/notebooks/65_step15_4_implement_minigpt.ipynb)已经讲过矩阵乘法、因果可见性与 softmax；本节新增的是 query 与 KV heads 数不同，以及它们如何成组对应。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def gqa_reference(q, k, v):
    # 本节限制为 equal-length causal self-attention，非 cache decode。
    B, Hq, T, d = q.shape
    if k.shape != v.shape or k.shape[0] != B or k.shape[2:] != (T, d):
        raise ValueError("expect matching K/V and equal-length self-attention")
    Hkv = k.shape[1]
    if Hkv <= 0 or Hq % Hkv:
        raise ValueError("Hq must be divisible by Hkv")
    repeated_k = mx.repeat(k, Hq // Hkv, axis=1)
    repeated_v = mx.repeat(v, Hq // Hkv, axis=1)
    scores = (q @ repeated_k.transpose(0, 1, 3, 2)) * (d ** -0.5)
    allowed = mx.arange(T)[:, None] >= mx.arange(T)[None, :]
    scores = mx.where(allowed, scores, float("-inf"))
    return mx.softmax(scores, axis=-1) @ repeated_v

## 5. 可选未来对照：两条路径从同一组原输入出发

下个单元按固定 arange 构造 q/k/v，没有随机采样。这里只说明未来要读什么；代码单元未执行，不填写预期误差值。

| 路径 | 传入的 K/V shape | 内部或接口上的区别 |
|---|---|---|
| `gqa_reference(q,k,v)` | (1,2,3,2) | 在函数局部创建 repeated_k/v，扩为 (1,4,3,2) |
| `mx.fast.scaled_dot_product_attention(q,k,v,...)` | 仍是原 (1,2,3,2) | 接口直接接收紧凑 GQA heads，不预先 repeat |

reference 没有把外面的 k/v 改写成 repeated_k/v；因此下一行 fast 仍收到原数组。这里不要求调用者另造一份物理副本，更不能把 reference 的展开中间量当成 fast 的输入。

固定 MLX v0.32.2 的 scale 是关键字参数，代码用 d=2 对应的 `scale=2**-0.5`。`mask="causal"` 在本节相等 T 的情况下与第 4.1 节下三角可见性对应。fast 支持紧凑 GQA 输入，不等于已经证明这次会选用某个特定 fused kernel 或得到加速。

In [ ]:
q = mx.arange(24, dtype=mx.float32).reshape(1, 4, 3, 2) / 24
k = mx.arange(12, dtype=mx.float32).reshape(1, 2, 3, 2) / 12
v = mx.arange(12, dtype=mx.float32).reshape(1, 2, 3, 2) / 6
reference = gqa_reference(q, k, v)
fast = mx.fast.scaled_dot_product_attention(q, k, v, scale=2 ** -0.5, mask="causal")
max_error = mx.max(mx.abs(reference - fast))
mx.eval(reference, fast, max_error)
print("output shapes:", reference.shape, fast.shape)
print("future observed max absolute difference:", max_error.item())

### 5.1 max absolute difference 怎么读？

`reference-fast` 逐元素比较同 shape 的输出；abs 去掉差值符号，`mx.max(...)` 未指定 axis，因此沿**全部 B/head/time/feature 元素**取最大值，得到标量：

$$
E_{\max}=\max_{b,h,t,c}
\left|\mathrm{reference}_{b,h,t,c}-\mathrm{fast}_{b,h,t,c}\right|.
$$

它不是平均误差、相对误差或正确率。未来先确认结果有限，再按 dtype、输入/输出尺度和用途选择容差；浮点运算顺序不同可能产生差异，不能要求差值必定为 0。NaN 也不能被当作“很小所以通过”。

即使未来这一个输入的最大绝对差很小，也只支持这组输入和这些参数下的数值接近；不证明所有形状、梯度、非方阵 decode、旧 checkpoint 转换都正确，更不说明峰值内存或速度。本次只保留显示差值的源码，没有执行它、没有实际差值或运行通过结论。

## 6. 保持架构与适用范围的边界

MLX v0.32.2 fast SDPA 支持 MHA/MQA/GQA，K/V 不应预先平铺到 Hq。它的 causal 对齐有自己的接口约定；本 reference 只覆盖相等 T 的 self-attention，不能把方阵例子当作已经实现增量 cache decode。已有生成循环也仍未实现 KV cache。

不同 KV head 数意味着 K/V projection 输出宽度不同。原 MHA 在数学上 Q/K/V 各有宽度 C；[实际源码](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/479bd59af3956c74cbad21787c7fdf8ff17eeda3/notebooks/65_step15_4_implement_minigpt.ipynb)用一个 C→3C 合并投影再 split。GQA 的 Q 宽度为 Hq·d，K/V 各为 Hkv·d，不能只修改 reshape 就得到这些新参数。把旧 K/V 权重平均成组也不是无损变换，不能宣称兼容原训练结果。

本节改变的是架构的 KV 共享关系。使用另一执行接口则是计算路径的选择，二者不能混成同一件事；本轮不改造已验收 MiniGPT，不处理 checkpoint，也不展开下一节算法。

## 7. 练习（不附答案）

1. Hq=8、Hkv=2 时画出 query→KV 映射，并列出 Q/K/V shapes。
2. 给定 N、B、T、d、dtype，写出 cache 容量随 Hkv 的比例关系。
3. 解释为什么 reference 中的 repeat 不应出现在最终缓存存储里。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 官方 API 阅读与完成边界

- [MLX v0.32.2 python/src/fast.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/fast.cpp)：SDPA 的 scale、GQA 原始 K/V 输入与 causal 对齐契约。
- [MLX v0.32.2 mlx/fast.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/fast.cpp)：按 KV head 与组内 query heads 展开连续分组。
- [MLX v0.32.2 python/src/ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)：repeat 的 axis/逐项重复、where、softmax 与归约接口。

本次保留原动机、容量公式与数值例子、两个代码单元和练习，聚焦补充分组/shape/mask/单行纸算和对照结果的解释。代码单元逐字未变，没有新增可执行入口。

严格 JSON/schema、合法唯一 ID、静态语法/AST 与文件范围检查不等于运行验证。全部代码单元 execution_count=null、outputs=[]；未执行 Notebook、MLX、训练、测试、实验、语料读/下载、checkpoint 或用户工作副本。没有差值、性能或峰值内存的实测记录。

22.4 累计 **1/2**（历史 0/2＋本次教学修订 1 次），其他历史计数保持。本轮停在 22.4，等待独立验收，不进入 22.5。